In [ ]:
from pathlib import Path

import seaborn as sns
from bonner.datasets.papale2025_tvsd import MONKEYS
from bonner.plotting import save_figure
from matplotlib import pyplot as plt

from lib.spectra import (
    bin_data,
    compute_spectra_with_n_fold_cross_validation,
    plot_spectra,
)
from lib.utilities import JOURNAL_MATPLOTLIBRC, extract_geometrically_spaced_bins

FIGURES_HOME = Path.cwd().parent / "figures"
FIGURES_HOME.mkdir(exist_ok=True, parents=True)

sns.set_theme(context="paper", style="ticks", rc=JOURNAL_MATPLOTLIBRC)

In [ ]:
from lib.datasets import split_by_repetition, tvsd

datasets = {
    monkey: split_by_repetition(
        tvsd.load_dataset(monkey=monkey, normalized=True),
        n_repetitions=1,
    )
    for monkey in MONKEYS
}

In [ ]:
dataset = mat73.loadmat(
    download(monkey=monkey, normalized=False),
    only_include=["ALLMAT", "tb"],
)

mapping = (
    np.squeeze(
        scipy.io.loadmat(download_electrode_mapping(monkey=monkey))["mapping"],
    )
    - 1
)
rois = _get_rois(monkey=monkey)
rois = np.array(rois)[np.argsort(mapping)]

stimuli = (
    pd.DataFrame(
        dataset["ALLMAT"][:, [1, 2, 4, 5]],
        columns=[
            "train_idx",
            "test_idx",
            "count",
            "day",
        ],
        dtype=np.uint32,
    )
    .assign(stimulus=lambda x: x["train_idx"] + x["test_idx"] - 1)
    .assign(training_set=lambda x: x["train_idx"] > 0)
    .drop(columns=["train_idx", "test_idx"])
)

dataset = xr.DataArray(
    name=f"{IDENTIFIER}.monkey={monkey}.normalized=False",
    data=mat73.loadmat(
        download(monkey=monkey, normalized=False),
        only_include=["ALLMUA"],
    )["ALLMUA"],
    dims=("electrode", "presentation", "time"),
    coords={
        "region": ("electrode", rois),
        "time": ("time", dataset["tb"]),
    }
    | {
        column: ("presentation", stimuli[column].to_numpy())
        for column in stimuli.columns
    },
)

In [ ]:
spectra = compute_spectra_with_n_fold_cross_validation(
    x_train=datasets["F"][0],
    y_train=datasets["N"][0],
    x_test=datasets["F"][0],
    y_test=datasets["N"][0],
    n_folds=8,
    n_permutations=5_000,
    n_bootstraps=0,
)


fig, ax = plt.subplots(figsize=(3, 3))

bin_edges, bin_centers = extract_geometrically_spaced_bins(
    start=1,
    stop=spectra.sizes["component"],
    density=3,
)

spectra = bin_data(
    spectra,
    bin_edges={"component": bin_edges},
    bin_centers={"component": bin_centers},
    dim="rank",
).expand_dims({"species": ["monkey"]})

plot_spectra(
    ax=ax,
    spectra=spectra,
    hue="species",
    palette="flare",
    hue_labels=["monkey"],
    marker="o",
    hide_insignificant=True,
    null_quantile=0.999,
)
ax.set_title("between-monkey", pad=10)
ax.set_ylabel("covariance")
ax.set_xlabel("rank")
ax.set_xlim(left=1, right=1_024)
ax.set_xscale("log")
ax.set_yscale("log")

save_figure(fig, filepath=FIGURES_HOME / "between-monkey.pdf")